# General Optimize

Create only `parameters.csv`, then train the unsupervised optimization pipeline with `optimize()`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from kkthn import KKTHardNet

TRAIN = {
    "epochs": 2,
    "batch_size": 8,
    "learning_rate": 1e-3,
    "train_frac": 0.8,
    "hidden_size": 16,
    "hidden_layers": 1,
    "seed": 42,
    "dtype": "float64",
    "print_every": 1,
}


## Generate Example Data

In [ ]:
data_dir = Path('optimization')
data_dir.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(5)
y1 = rng.uniform(0.0, 1.0, size=1000)
y3 = rng.uniform(-1.0, 1.0, size=1000)
mask = y1**2 + y3**2 <= 1.6
y1 = y1[mask][:1000]
y3 = y3[mask][:1000]
y2 = rng.uniform(-0.5, 0.5, size=y1.size)
a0, a1 = 1.0, -1.0
x1 = a0 * y1 + y2
x2 = y2 - a1 * y3

pd.DataFrame({'x1': x1, 'x2': x2}).to_csv(data_dir / 'parameters.csv', index=False)
data_dir


## Build And Optimize

In [ ]:
model = KKTHardNet(name='general_optimize', train=TRAIN)
x = model.add_parameter(['x1', 'x2'])
theta = model.add_inverse_parameter(['a0', 'a1'], init_value=[1.0, -1.0])
y = model.add_variable(['y1', 'y2', 'y3'])
model.objective = 0.5 * (y.y1**2 + y.y2**2 + y.y3**2)
model.constraints.add(
    theta.a0 * y.y1 + y.y2 - x.x1 == 0,
    y.y2 - theta.a1 * y.y3 - x.x2 == 0,
    y.y1**2 + y.y3**2 <= 2.0,
    y.y1 >= 0,
)
model.dataset(parameters=data_dir / 'parameters.csv')
result = model.optimize()
result['metadata_path']


In [ ]:
KKTHardNet().load(result['metadata_path']).predict([1.0, 0.25])
